<a href="https://colab.research.google.com/github/MatteoBaraldi/Machine-Learning-for-Bioengineering/blob/main/MOD-1/exams-templates/Template_BSPML_20260608.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

DO NOT MODIFY THE NEXT CELL (PENALTY of 0.5 POINTS)

In [ ]:
from sklearn.datasets import make_classification

# generate synthetic biomedical data

X,y = make_classification(n_samples=500, n_features=20, n_informative=5, n_redundant=3, weights=[0.6, 0.4], class_sep=1.0, random_state=42)



PLACE BELOW YOUR CODE

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold, cross_validate, train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error
import numpy as np

SEED = 65

outer_n_folds = 5

inner_n_folds = 5

C = [0.01, 0.1, 1, 10, 100]

outer_cv = KFold(n_splits=outer_n_folds, shuffle=True, random_state=SEED)

inner_cv = KFold(n_splits=inner_n_folds, shuffle=True, random_state=SEED)

clf = LogisticRegression(max_iter=1000)

p_grid = [{'C': C}]

clf_gs = GridSearchCV(clf, param_grid=p_grid, cv=inner_cv, refit='roc_auc', scoring='roc_auc', verbose=4)

nested_score = cross_validate(clf_gs, X=X, y=y, cv=outer_cv, return_train_score=True, return_estimator=True, scoring= ['accuracy','roc_auc'])

acc = nested_score['test_accuracy']
auc = nested_score['test_roc_auc']

print(f"Accuracy : mean = {acc.mean():.4f}  std = {acc.std():.4f}")
print(f"ROC AUC  : mean = {auc.mean():.4f}  std = {auc.std():.4f}")

# utile per il commento: quale C è stato scelto in ogni fold esterno
for i, est in enumerate(nested_score['estimator'], 1):
    print(f"Fold {i}: C = {est.best_params_['C']}")



Fitting 5 folds for each of 5 candidates, totalling 25 fits
[CV 1/5] END ............................C=0.01;, score=0.924 total time=   0.1s
[CV 2/5] END ............................C=0.01;, score=0.809 total time=   0.1s
[CV 3/5] END ............................C=0.01;, score=0.934 total time=   0.1s
[CV 4/5] END ............................C=0.01;, score=0.970 total time=   0.1s
[CV 5/5] END ............................C=0.01;, score=0.882 total time=   0.1s
[CV 1/5] END .............................C=0.1;, score=0.939 total time=   0.1s
[CV 2/5] END .............................C=0.1;, score=0.835 total time=   0.0s
[CV 3/5] END .............................C=0.1;, score=0.939 total time=   0.1s
[CV 4/5] END .............................C=0.1;, score=0.950 total time=   0.0s
[CV 5/5] END .............................C=0.1;, score=0.908 total time=   0.0s
[CV 1/5] END ...............................C=1;, score=0.937 total time=   0.0s
[CV 2/5] END ...............................C=1;,

1) With a single CV the same data does double duty: it picks C and it estimates
performance. That inflates the score, since you end up rewarding the C that happens to suit those particular folds. Nested CV keeps the two jobs apart. The inner loop chooses C, the outer loop scores the result on folds that played no part in the choice, so the outer average reflects how the whole procedure would behave on new data.

2) ROC AUC, because the data is imbalanced (60/40) and accuracy gives too much
credit for getting the common class right. What you actually want to know is whether the model can rank patients by risk, and that is what ROC AUC measures, regardless of how common each class is.

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV

SEED = 65
C = [0.01, 0.1, 1, 10, 100]
weight_settings = [[0.6, 0.4], [0.8, 0.2], [0.96, 0.04]]

results = {}

for w in weight_settings:
    X, y = make_classification(n_samples=500, n_features=20, n_informative=5,
                               n_redundant=3, weights=w, class_sep=1.0,
                               random_state=42)

    clf = LogisticRegression(max_iter=1000)
    p_grid = [{'C': C}]

    # Inner loop: 5-fold CV to tune C, optimizing ROC AUC
    inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    # Outer loop: 5-fold CV to estimate generalization performance
    outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

    clf_gs = GridSearchCV(clf, param_grid=p_grid, cv=inner_cv, scoring='roc_auc')
    nested = cross_validate(clf_gs, X=X, y=y, cv=outer_cv,
                            scoring=['accuracy', 'roc_auc'])

    acc, auc = nested['test_accuracy'], nested['test_roc_auc']
    results[tuple(w)] = (acc, auc)

    print(f"weights = {w}")
    print(f"  mean Accuracy = {acc.mean():.4f}  (std {acc.std():.4f})")
    print(f"  mean ROC AUC  = {auc.mean():.4f}  (std {auc.std():.4f})")
    print()

# Comparison table
print("Summary")
print(f"{'weights':<14}{'mean_acc':>12}{'mean_auc':>12}")
for w, (acc, auc) in results.items():
    print(f"{str(list(w)):<14}{acc.mean():>12.4f}{auc.mean():>12.4f}")

weights = [0.6, 0.4]
  mean Accuracy = 0.8660  (std 0.0150)
  mean ROC AUC  = 0.9151  (std 0.0191)

weights = [0.8, 0.2]
  mean Accuracy = 0.8980  (std 0.0133)
  mean ROC AUC  = 0.8902  (std 0.0262)

weights = [0.96, 0.04]
  mean Accuracy = 0.9560  (std 0.0049)
  mean ROC AUC  = 0.7775  (std 0.0865)

Summary
weights           mean_acc    mean_auc
[0.6, 0.4]          0.8660      0.9151
[0.8, 0.2]          0.8980      0.8902
[0.96, 0.04]        0.9560      0.7775


1) The two metrics pull in opposite directions. As the minority class shrinks,
accuracy climbs (0.87 → 0.90 → 0.96) while ROC AUC drops (0.92 → 0.89 → 0.78). The rising accuracy is an artifact: at 96/4, guessing "healthy" every time already scores about 0.954, so the model's 0.956 is barely an improvement. ROC AUC tells the real story. With only few sick patients to learn from, the model separates the classes noticeably worse.

2) ROC AUC. It is built from sensitivity and specificity, measured inside each class separately, and it averages over all decision thresholds. That is why it stays stable no matter how skewed the classes are. Accuracy has no such protection: it is a raw count of correct predictions, so the majority class dominates it as soon as the split gets lopsided.

3) No. The patients who matter most in diagnosis are usually the rare ones, and
accuracy barely registers them. A 96% accurate model that quietly labels nearly
everyone "healthy" misses almost every real case, which in a clinical setting means missed diagnoses. You need metrics that watch the minority class directly, like recall and F1, with the threshold set by how costly a false negative is versus a false positive.